[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sergiovillanueva/Modelos_Fundacionales/blob/main/1_OD.ipynb)

**Día 1 · Cuaderno 1 de 3** · Acompaña al tema 1 de la web: [Detección de objetos](https://datamecum.sergiovillanuevalopez.com/temas/01-deteccion/)

**En clase:** hasta *Antes y después*. **Extra**, si da tiempo: quedarse solo con algunas clases.

# Detección de objetos con RF-DETR

Detectar objetos es encontrar qué hay en una imagen y dónde: el modelo devuelve una caja por objeto, con su clase y su confianza.

- **Clasificación**: "en esta imagen hay un gato". Una etiqueta para toda la imagen.
- **Detección**: "hay tres gatos, aquí, aquí y aquí". Una caja y una etiqueta por objeto.

Usamos **RF-DETR**, de Roboflow: un detector moderno, con licencia Apache 2.0, que viene ya entrenado con las 80 clases de COCO. Primero lo usamos tal cual y al final lo entrenamos con un dataset pequeño de baloncesto para que aprenda clases nuevas.

**Antes de empezar**

1. Activa la GPU: menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.
2. Ejecuta las celdas en orden, de arriba abajo, con `Mayúsculas + Intro`.
3. La primera celda instala el paquete `rfdetr` (tarda un minuto).

Este cuaderno es tuyo: al abrirlo desde el enlace se crea una copia que solo ves tú, y puedes cambiar lo que quieras sin estropear el original.

## De dónde viene RF-DETR

- **R-CNN, Fast R-CNN, Faster R-CNN** (2014-2015): dos etapas. Primero proponen zonas y luego las clasifican. Precisos, pero lentos.
- **YOLO** (desde 2016): una sola pasada por la imagen, muy rápido. Las versiones de Ultralytics (YOLOv8, YOLO11...) tienen licencia AGPL-3.0: en un producto, publicas tu código o pagas.
- **DETR** (2020): el primer detector con transformers. Da directamente las cajas, sin anchors ni NMS.
- **RF-DETR** (2025): un DETR rápido, con licencia Apache 2.0. Se puede usar en un producto sin pagar nada.

## Preparación

In [ ]:
# Instala rfdetr con lo necesario para entrenar, si no está ya
try:
    import rfdetr, pytorch_lightning  # noqa: F401
except ImportError:
    %pip install -q "rfdetr[train]==1.12.0"

import requests
import torch
import matplotlib.pyplot as plt
from collections import Counter
from io import BytesIO
from pathlib import Path
from PIL import Image
from rfdetr import RFDETRMedium

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando:", device)
if device == "cpu":
    print("Aviso: no hay GPU. Actívala en Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4.")

ASSETS = "https://raw.githubusercontent.com/sergiovillanueva/Modelos_Fundacionales/main/assets/"


def cargar_imagen(nombre_o_url):
    """Abre una imagen del curso (de assets/ si existe, si no del repositorio) o de una URL."""
    local = Path("assets") / Path(nombre_o_url).name
    if local.exists():
        return Image.open(local).convert("RGB")
    url = nombre_o_url if nombre_o_url.startswith("http") else ASSETS + nombre_o_url
    respuesta = requests.get(url, timeout=30)
    respuesta.raise_for_status()
    return Image.open(BytesIO(respuesta.content)).convert("RGB")


image_cars = cargar_imagen("cars.jpg")
image_person = cargar_imagen("person_cars.jpg")
image_bananas = cargar_imagen("bananas.jpg")
image_fruits = cargar_imagen("fruits.jpg")
print("Imágenes cargadas")

## Cargar el detector

`RFDETRMedium()` descarga los pesos la primera vez (unos 130 MB) y los guarda para las siguientes. Hay tamaños de *nano* a *2XL*: más grande es más preciso y más lento. *Medium* es un buen equilibrio.

El modelo viene entrenado con **COCO**, un dataset de 80 clases de objetos comunes: personas, coches, animales, muebles, comida...

In [ ]:
modelo_coco = RFDETRMedium()
print("RF-DETR cargado")

## Qué devuelve

`predict` devuelve las detecciones de la imagen. Por cada objeto:

- `xyxy`: la caja en píxeles, esquina superior izquierda (x1, y1) e inferior derecha (x2, y2),
- `confidence`: la confianza, de 0 a 1,
- `data["class_name"]`: el nombre de la clase.

In [ ]:
detecciones = modelo_coco.predict(image_person, threshold=0.5)

print(f"{len(detecciones)} objetos. Los tres primeros:")
for caja, confianza, clase in list(zip(detecciones.xyxy, detecciones.confidence, detecciones.data["class_name"]))[:3]:
    print(f"  {clase}: {confianza:.2f} en {caja.round().astype(int).tolist()}")

## Dibujar las detecciones

Con las cajas en la mano, dibujar es pintar un rectángulo por objeto. Cada clase sale de un color.

In [ ]:
def dibujar(imagen, detecciones, titulo=""):
    """Dibuja cada caja con su clase y su confianza. Un color por clase."""
    colores = {}
    plt.figure(figsize=(12, 8))
    plt.imshow(imagen)
    eje = plt.gca()
    for (x1, y1, x2, y2), confianza, clase in zip(detecciones.xyxy, detecciones.confidence, detecciones.data["class_name"]):
        color = colores.setdefault(clase, plt.cm.tab10(len(colores) % 10))
        eje.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, linewidth=3))
        eje.text(x1, y1, f"{clase} {confianza:.2f}", color="white", fontsize=10, va="bottom",
                 bbox=dict(facecolor=color, edgecolor="none", pad=1.5))
    plt.title(titulo)
    plt.axis("off")
    plt.show()


def detectar(imagen, umbral=0.5, modelo=None):
    """Detecta con el umbral indicado, dibuja y devuelve las detecciones."""
    modelo = modelo or modelo_coco
    detecciones = modelo.predict(imagen, threshold=umbral)
    dibujar(imagen, detecciones, f"{len(detecciones)} objetos con umbral {umbral}")
    return detecciones


_ = detectar(image_person)

## El umbral de confianza

El umbral decide qué detecciones se quedan:

- **Alto (0,7 a 0,9)**: solo lo que el modelo tiene muy claro. Menos falsas alarmas (más precisión), pero se pierden objetos.
- **Bajo (0,2 a 0,4)**: aparecen más objetos, también alguno que no lo es (más recall, menos precisión).

Mueve el deslizador y busca el punto en el que no sobran cajas ni faltan objetos.

In [ ]:
from ipywidgets import interact, FloatSlider

@interact(umbral=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.5, description="Umbral"))
def probar_umbral(umbral):
    detectar(image_person, umbral)

## Qué sabe detectar

El modelo conoce las 80 clases de COCO:

In [ ]:
print(len(modelo_coco.class_names), "clases:")
print(", ".join(modelo_coco.class_names))

## Aplicación: contar objetos

Contar es de lo más pedido en la industria: piezas en una bandeja, coches en un aparcamiento, personas en una zona. Con las detecciones basta con contar nombres.

In [ ]:
def contar(imagen, umbral=0.5):
    """Cuenta los objetos de cada clase."""
    detecciones = detectar(imagen, umbral)
    for clase, cantidad in Counter(detecciones.data["class_name"]).most_common():
        print(f"{clase}: {cantidad}")


contar(image_bananas)

## Ejercicio: tu propia imagen

Prueba el detector con una foto tuya: tu mesa (portátil, ratón, taza...), la calle (coches, personas, semáforos) o cualquier cosa que esté entre las 80 clases de COCO. Quita el `#` de la opción que quieras.

In [ ]:
# Opción 1: desde una URL
# mi_imagen = cargar_imagen("https://...")

# Opción 2: subiendo un archivo desde tu ordenador (solo en Colab)
# from google.colab import files
# subido = files.upload()
# mi_imagen = Image.open(next(iter(subido))).convert("RGB")

# _ = detectar(mi_imagen, umbral=0.4)

## El límite: solo conoce 80 clases

Un modelo entrenado con COCO detecta bien personas, coches o perros, pero no sabe nada de lo que no estaba en COCO: tornillos, defectos de fabricación, piezas de tu empresa... o jugadores, árbitros y canastas de un partido de baloncesto. Para él, todos son `person`.

## Ajuste fino: enseñarle las clases nuevas

El **ajuste fino** (fine-tuning) parte del modelo ya entrenado con COCO y lo sigue entrenando con nuestras imágenes. Como ya sabe ver bordes, formas y objetos, aprende las clases nuevas con pocos datos.

Usamos un dataset muy pequeño de [Roboflow Universe](https://universe.roboflow.com/roboflow-universe-projects/basketball-players-fy4c2): **15 fotos** de partidos de baloncesto (10 para entrenar, 3 para validar y 2 para probar), con jugadores, árbitros, balón, canasta y los rótulos del marcador. Licencia CC BY 4.0.

Viene en formato COCO, que es el que entiende `rfdetr`: una carpeta por partición (`train`, `valid`, `test`) con las fotos y un fichero `_annotations.coco.json` con las cajas.

In [ ]:
import json
import zipfile

carpeta_datos = Path("baloncesto")
if not carpeta_datos.exists():
    local = Path("assets") / "baloncesto.zip"
    origen = local if local.exists() else BytesIO(requests.get(ASSETS + "baloncesto.zip", timeout=60).content)
    zipfile.ZipFile(origen).extractall(".")

anotaciones = json.loads((carpeta_datos / "train" / "_annotations.coco.json").read_text())
nombres = {c["id"]: c["name"] for c in anotaciones["categories"]}
print("Clases:", [n for i, n in nombres.items() if i > 0])   # la 0 es la categoría que agrupa a las demás

# Una foto de entrenamiento con sus cajas anotadas a mano (en COCO: x, y, ancho, alto)
foto = anotaciones["images"][0]
imagen = Image.open(carpeta_datos / "train" / foto["file_name"]).convert("RGB")
plt.figure(figsize=(12, 7))
plt.imshow(imagen)
for a in anotaciones["annotations"]:
    if a["image_id"] == foto["id"]:
        x, y, w, h = a["bbox"]
        plt.gca().add_patch(plt.Rectangle((x, y), w, h, fill=False, color="lime", linewidth=2))
        plt.text(x, y, nombres[a["category_id"]], color="black", fontsize=8, va="bottom",
                 bbox=dict(facecolor="lime", edgecolor="none", pad=1))
plt.title("Anotación hecha a mano")
plt.axis("off")
plt.show()

### Antes de entrenar: qué ve el modelo de COCO

En una foto de prueba, que no se usa para entrenar, el modelo de COCO encuentra personas y, con suerte, el balón (`sports ball`). No distingue jugadores de árbitros ni sabe qué es una canasta.

In [ ]:
foto_prueba = sorted((carpeta_datos / "test").glob("*.jpg"))[0]
imagen_prueba = Image.open(foto_prueba).convert("RGB")
_ = detectar(imagen_prueba, umbral=0.5)

### Entrenar

Una línea: `train` con la carpeta del dataset y el número de épocas (pasadas completas por las fotos de entrenamiento). Durante el entrenamiento verás tablas con la mAP en las fotos de validación: tiene que ir subiendo.

En una GPU T4 de Colab tarda unos minutos. Los resultados se guardan en la carpeta `salida-baloncesto`.

In [ ]:
modelo_baloncesto = RFDETRMedium()
modelo_baloncesto.train(
    dataset_dir=str(carpeta_datos),
    epochs=20,
    batch_size=4,
    grad_accum_steps=1,
    num_workers=0,            # carga las fotos en el mismo proceso: con 15 fotos sobra, y funciona igual en Windows
    output_dir="salida-baloncesto",
)

## Antes y después

El mejor modelo del entrenamiento queda en `salida-baloncesto/checkpoint_best_total.pth`. Se carga pasándole esa ruta a `RFDETRMedium` y se usa igual que el de COCO.

In [ ]:
modelo_nuevo = RFDETRMedium(pretrain_weights="salida-baloncesto/checkpoint_best_total.pth")
print("Clases del modelo nuevo:", modelo_nuevo.class_names)

_ = detectar(imagen_prueba, umbral=0.5, modelo=modelo_coco)
_ = detectar(imagen_prueba, umbral=0.3, modelo=modelo_nuevo)

Con solo 10 fotos de entrenamiento el modelo ya distingue jugadores, árbitros y balón, cosas que el modelo de COCO no conocía. No es perfecto: alguna caja sobra o falta, sobre todo en los rótulos pequeños del marcador. Con más fotos y más épocas mejora, y para un sistema real se mide con un conjunto de prueba antes de usarlo.

**Con tus propios datos** el proceso es el mismo: anota tus fotos en [Roboflow](https://roboflow.com) o [CVAT](https://www.cvat.ai), exporta en formato COCO con las carpetas `train`, `valid` y `test`, y cambia la ruta de `dataset_dir`.

## Extra: quedarse solo con algunas clases

Si vamos bien de tiempo lo hacemos juntos en clase; si no, queda para casa. Completa los huecos `___` y quita el `#` de las líneas de prueba.

Muchas veces solo interesan unas clases: los coches de un aparcamiento o las personas de una zona. Las detecciones se pueden filtrar como una lista, con una máscara de verdadero o falso.

In [ ]:
def detectar_solo(imagen, clases_permitidas, umbral=0.5):
    """Detecta objetos y se queda solo con los de las clases de la lista."""
    detecciones = modelo_coco.predict(imagen, threshold=umbral)
    mascara = [clase in ___ for clase in detecciones.data["class_name"]]   # hueco 1: ¿con qué lista comparamos?
    elegidas = detecciones[mascara]
    dibujar(imagen, elegidas, f"Solo {', '.join(clases_permitidas)}: {len(elegidas)}")
    return elegidas


# Hueco 2: solo los coches del aparcamiento (el nombre está en la lista de clases de arriba)
# _ = detectar_solo(image_cars, ["___"])

# Hueco 3: en la foto del paso de cebra, la persona y los coches a la vez
# _ = detectar_solo(image_person, ["___", "___"])

## Resumen

- La detección devuelve una caja, una clase y una confianza por objeto.
- RF-DETR se usa con dos líneas: `RFDETRMedium()` y `predict`.
- El umbral de confianza equilibra precisión y recall.
- Contar objetos es contar los nombres de las detecciones.
- Un modelo entrenado con COCO solo conoce sus 80 clases.
- Con ajuste fino aprende clases nuevas: un dataset en formato COCO y `train`.

**Siguiente:** cuaderno 2, Introducción a Hugging Face. Antes, el tema 2 de la web. [Abrir el cuaderno en Colab](https://colab.research.google.com/github/sergiovillanueva/Modelos_Fundacionales/blob/main/2_HF_Intro.ipynb).